# Evaluation on July-December 2010

AI SCI 290A

This notebook records the July-December comparison for the fixed models listed below. The model comparisons are organized by topic in notebooks 02-05. This run does not evaluate every spatial, neural-network, numerical precipitation or hot-day model in those notebooks.

## 1. Models fixed for this run

| Target | Model and inputs |
| --- | --- |
| Tomorrow's temperature | Linear regression: today's temperature, its 3-day and 7-day averages, u, and v |
| Tomorrow's u | Linear regression: today's u and v |
| Tomorrow's v | Linear regression: today's u and v, their 3-day and 7-day averages, and pressure |
| Tomorrow's positive tp label | Logistic regression: today's tp, temperature, pressure, u, and v; cutoff 0.5 |

I compare temperature and wind with persistence using MAE and RMSE. For the tp label, I compare with always-zero and persistence using accuracy, precision, recall, F1, and a confusion matrix.

I fit each model once using target dates before July 2010. I do not use these test scores to choose new inputs or adjust the cutoff.

### Limits of this test

I already saw a temperature test on July-December 2010 in an earlier version. This period is not a completely unseen test for the whole project. I use it here as a final comparison with the current choices fixed. A fresh independent test would need another unused period.

The precipitation label means the stored tp value is above zero. It is not a confirmed full-day rain label or rainfall amount. These six test months also do not cover a full year.

Each prediction is made after today's daily data are available. Earlier observations within the test period can be used for the next day's inputs and rolling averages. The model coefficients stay fixed.

## 2. Load the original data

In [1]:
from pathlib import Path
import glob
import xarray as xr
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data" / "data"

In [2]:
temp_files = sorted(glob.glob(f"{DATA_DIR}/**/2m_temperature*.nc", recursive=True))
ds_temp = xr.open_mfdataset(temp_files, combine="by_coords")
temp_series = (ds_temp["t2m"].sel(latitude=33.75, longitude=-117.75, method="nearest") - 273.15).to_series()
print("temp files:", len(temp_files))

temp files: 12


In [3]:
u_files = sorted(glob.glob(f"{DATA_DIR}/**/10m_u_component_of_wind*.nc", recursive=True))
ds_u = xr.open_mfdataset(u_files, combine="by_coords")
u_series = (ds_u["u10"].sel(latitude=33.75, longitude=-117.75, method="nearest")).to_series()
print("u files:", len(u_files))

u files: 12


In [4]:
v_files = sorted(glob.glob(f"{DATA_DIR}/**/10m_v_component_of_wind*.nc", recursive=True))
ds_v = xr.open_mfdataset(v_files, combine="by_coords")
v_series = (ds_v["v10"].sel(latitude=33.75, longitude=-117.75, method="nearest")).to_series()
print("v files:", len(v_files))

v files: 12


In [5]:
pressure_files = sorted(glob.glob(f"{DATA_DIR}/**/mean_sea_level_pressure*.nc", recursive=True))
ds_pressure = xr.open_mfdataset(pressure_files, combine="by_coords")
pressure_series = (ds_pressure["msl"].sel(latitude=33.75, longitude=-117.75, method="nearest") / 100).to_series()
print("pressure files:", len(pressure_files))

pressure files: 12


In [6]:
tp_files = sorted(glob.glob(f"{DATA_DIR}/**/total_precipitation*.nc", recursive=True))
ds_tp = xr.open_mfdataset(tp_files, combine="by_coords")
tp_series = (ds_tp["tp"].sel(latitude=33.75, longitude=-117.75, method="nearest")).to_series()
print("tp files:", len(tp_files))

tp files: 12


## 3. Prepare the daily table

I match the dates and check that the daily series is complete before shifting the targets. The averages use today and earlier days only.

In [7]:
assert temp_series.index.equals(u_series.index)
assert temp_series.index.equals(v_series.index)
assert temp_series.index.equals(pressure_series.index)
assert temp_series.index.equals(tp_series.index)

weather = pd.DataFrame({
    "today_temp": temp_series, "today_u": u_series, "today_v": v_series,
    "pressure_hpa": pressure_series, "today_tp": tp_series
})
assert weather.index.is_unique and weather.index.is_monotonic_increasing
assert weather.index.equals(pd.date_range("2009-01-01", "2010-12-31"))
assert not weather.isna().any().any()
print("Dates match; 730 daily rows with no missing values.")

Dates match; 730 daily rows with no missing values.


In [8]:
weather["temp_3day"] = weather["today_temp"].rolling(3).mean()
weather["temp_7day"] = weather["today_temp"].rolling(7).mean()
weather["u_3day"] = weather["today_u"].rolling(3).mean()
weather["u_7day"] = weather["today_u"].rolling(7).mean()
weather["v_3day"] = weather["today_v"].rolling(3).mean()
weather["v_7day"] = weather["today_v"].rolling(7).mean()

weather["tomorrow_temp"] = weather["today_temp"].shift(-1)
weather["tomorrow_u"] = weather["today_u"].shift(-1)
weather["tomorrow_v"] = weather["today_v"].shift(-1)
weather["tomorrow_tp"] = weather["today_tp"].shift(-1)
weather["target_date"] = weather.index + pd.Timedelta(days=1)

temp_features = ["today_temp", "temp_3day", "temp_7day", "today_u", "today_v"]
u_features = ["today_u", "today_v"]
v_features = ["today_u", "today_v", "u_3day", "u_7day", "v_3day", "v_7day", "pressure_hpa"]
signal_features = ["today_tp", "today_temp", "pressure_hpa", "today_u", "today_v"]

## 4. Fix the training and test dates

For temperature and wind, I use the common rows with a full 7-day history, as in the last input comparisons. The precipitation model needs no rolling history, so it keeps the first six days. Each model and its baseline use the same test dates.

Training includes the earlier validation months. I fit scaling again on these training rows only.

In [9]:
regression_data = weather.dropna(subset=temp_features + v_features + ["tomorrow_temp", "tomorrow_u", "tomorrow_v"]).copy()
signal_data = weather.dropna(subset=signal_features + ["tomorrow_tp"]).copy()
signal_data["today_signal"] = (signal_data["today_tp"] > 0).astype(int)
signal_data["tomorrow_signal"] = (signal_data["tomorrow_tp"] > 0).astype(int)

train_reg = regression_data[regression_data["target_date"] < "2010-07-01"].copy()
test_reg = regression_data[
    (regression_data["target_date"] >= "2010-07-01") &
    (regression_data["target_date"] < "2011-01-01")
].copy()
train_signal = signal_data[signal_data["target_date"] < "2010-07-01"].copy()
test_signal = signal_data[
    (signal_data["target_date"] >= "2010-07-01") &
    (signal_data["target_date"] < "2011-01-01")
].copy()

assert train_reg["target_date"].max() < test_reg["target_date"].min()
assert train_signal["target_date"].max() < test_signal["target_date"].min()
assert test_reg["target_date"].equals(test_signal["target_date"])
assert len(test_reg) == 184
print("Regression training rows:", len(train_reg))
print("Classification training rows:", len(train_signal))
print("Test rows for each target:", len(test_reg))
print("Test target dates:", test_reg["target_date"].min(), "to", test_reg["target_date"].max())

Regression training rows: 539
Classification training rows: 545
Test rows for each target: 184
Test target dates: 2010-07-01 00:00:00 to 2010-12-31 00:00:00


## 5. Run the final evaluation

I use the model choices and settings written above. I fit on target dates before July 2010 and evaluate July-December without changing the inputs or cutoff.

In [10]:
final_temp_model = LinearRegression()
final_temp_model.fit(train_reg[temp_features], train_reg["tomorrow_temp"])
final_u_model = LinearRegression()
final_u_model.fit(train_reg[u_features], train_reg["tomorrow_u"])
final_v_model = LinearRegression()
final_v_model.fit(train_reg[v_features], train_reg["tomorrow_v"])

test_reg["temp_prediction"] = final_temp_model.predict(test_reg[temp_features])
test_reg["u_prediction"] = final_u_model.predict(test_reg[u_features])
test_reg["v_prediction"] = final_v_model.predict(test_reg[v_features])

### Temperature: compare with persistence

In [11]:
temp_baseline_error = test_reg["today_temp"] - test_reg["tomorrow_temp"]
temp_model_error = test_reg["temp_prediction"] - test_reg["tomorrow_temp"]
temp_test_results = pd.DataFrame({
    "Model": ["Persistence", "Linear regression"],
    "Unit": ["C", "C"],
    "MAE": [temp_baseline_error.abs().mean(), temp_model_error.abs().mean()],
    "RMSE": [((temp_baseline_error ** 2).mean()) ** 0.5, ((temp_model_error ** 2).mean()) ** 0.5]
})
temp_test_results.round(4)

,Model,Unit,MAE,RMSE
0,Persistence,C,0.9983,1.3152
1,Linear regression,C,0.8962,1.1816


### u component: compare with persistence

In [12]:
u_baseline_error = test_reg["today_u"] - test_reg["tomorrow_u"]
u_model_error = test_reg["u_prediction"] - test_reg["tomorrow_u"]
u_test_results = pd.DataFrame({
    "Model": ["Persistence", "Linear regression"],
    "Unit": ["m/s", "m/s"],
    "MAE": [u_baseline_error.abs().mean(), u_model_error.abs().mean()],
    "RMSE": [((u_baseline_error ** 2).mean()) ** 0.5, ((u_model_error ** 2).mean()) ** 0.5]
})
u_test_results.round(4)

,Model,Unit,MAE,RMSE
0,Persistence,m/s,0.5319,0.8279
1,Linear regression,m/s,0.4684,0.6783


### v component: compare with persistence

In [13]:
v_baseline_error = test_reg["today_v"] - test_reg["tomorrow_v"]
v_model_error = test_reg["v_prediction"] - test_reg["tomorrow_v"]
v_test_results = pd.DataFrame({
    "Model": ["Persistence", "Linear regression"],
    "Unit": ["m/s", "m/s"],
    "MAE": [v_baseline_error.abs().mean(), v_model_error.abs().mean()],
    "RMSE": [((v_baseline_error ** 2).mean()) ** 0.5, ((v_model_error ** 2).mean()) ** 0.5]
})
v_test_results.round(4)

,Model,Unit,MAE,RMSE
0,Persistence,m/s,0.4974,0.7035
1,Linear regression,m/s,0.4503,0.6359


### Precipitation signal

I use the same default logistic settings and 0.5 cutoff as before. The scaler learns from the final training rows only.

In [14]:
final_scaler = StandardScaler()
X_train_signal = final_scaler.fit_transform(train_signal[signal_features])
X_test_signal = final_scaler.transform(test_signal[signal_features])
final_signal_model = LogisticRegression(max_iter=1000)
final_signal_model.fit(X_train_signal, train_signal["tomorrow_signal"])

test_signal["always_zero"] = 0
test_signal["persistence"] = test_signal["today_signal"]
test_signal["probability"] = final_signal_model.predict_proba(X_test_signal)[:, 1]
test_signal["logistic"] = (test_signal["probability"] >= 0.5).astype(int)
y_test_signal = test_signal["tomorrow_signal"]
print("Actual label counts:")
print(y_test_signal.value_counts().sort_index())

Actual label counts:
tomorrow_signal
0    106
1     78
Name: count, dtype: int64


In [15]:
always_zero_test_scores = {
    "Model": "Always zero",
    "Accuracy": accuracy_score(y_test_signal, test_signal["always_zero"]),
    "Precision": precision_score(y_test_signal, test_signal["always_zero"], zero_division=0),
    "Recall": recall_score(y_test_signal, test_signal["always_zero"], zero_division=0),
    "F1": f1_score(y_test_signal, test_signal["always_zero"], zero_division=0)
}

In [16]:
persistence_test_scores = {
    "Model": "Persistence",
    "Accuracy": accuracy_score(y_test_signal, test_signal["persistence"]),
    "Precision": precision_score(y_test_signal, test_signal["persistence"], zero_division=0),
    "Recall": recall_score(y_test_signal, test_signal["persistence"], zero_division=0),
    "F1": f1_score(y_test_signal, test_signal["persistence"], zero_division=0)
}

In [17]:
logistic_test_scores = {
    "Model": "Logistic",
    "Accuracy": accuracy_score(y_test_signal, test_signal["logistic"]),
    "Precision": precision_score(y_test_signal, test_signal["logistic"], zero_division=0),
    "Recall": recall_score(y_test_signal, test_signal["logistic"], zero_division=0),
    "F1": f1_score(y_test_signal, test_signal["logistic"], zero_division=0)
}

In [18]:
signal_test_results = pd.DataFrame([always_zero_test_scores, persistence_test_scores, logistic_test_scores])
signal_test_results.round(4)

,Model,Accuracy,Precision,Recall,F1
0,Always zero,0.5761,0.0000,0.0000,0.000
1,Persistence,0.6957,0.6410,0.6410,0.641
2,Logistic,0.7065,0.7857,0.4231,0.550


In [19]:
pd.DataFrame(
    confusion_matrix(y_test_signal, test_signal["logistic"], labels=[0, 1]),
    index=["Actual zero", "Actual positive"],
    columns=["Predicted zero", "Predicted positive"]
)

,Predicted zero,Predicted positive
Actual zero,97,9
Actual positive,45,33


## 6. What I found

On the 184 July-December 2010 days, the temperature and wind models have lower MAE and RMSE than persistence.

| Target | Persistence MAE | Model MAE | Persistence RMSE | Model RMSE |
| --- | ---: | ---: | ---: | ---: |
| Temperature (C) | 0.9983 | 0.8962 | 1.3152 | 1.1816 |
| u (m/s) | 0.5319 | 0.4684 | 0.8279 | 0.6783 |
| v (m/s) | 0.4974 | 0.4503 | 0.7035 | 0.6359 |

The simple models help on this period, but this does not mean they will work equally well in another year. The temperature model here includes wind, so it is not the same as the earlier temperature-only version.

### The precipitation result is mixed

Logistic regression has slightly higher accuracy than persistence (0.7065 versus 0.6957) and higher precision (0.7857 versus 0.6410). But recall is lower (0.4231 versus 0.6410), and F1 falls to 0.5500 compared with 0.6410 for persistence.

There are 78 positive targets. Logistic regression finds 33 and misses 45, with 9 false alarms. Its positive predictions are more often right, but it misses many positive days. The validation F1 improvement did not carry over to this period.

I keep this result rather than changing the cutoff after seeing it. The label still means positive stored tp, not a confirmed full-day rain event. The always-zero model's precision is undefined and is displayed as zero in the table.

### What I can say about the project

Recent weather helped the simple temperature and wind models beat persistence here. The precipitation classifier did not clearly beat persistence: the answer depends on whether finding more positive days or avoiding false alarms matters more.

I kept the chosen inputs and settings fixed for this run. However, I had already seen a temperature test on these dates, so this is not a completely unseen test for the whole project. A stronger check would use another unused period. The two years of data and uncertain precipitation sampling are still limits.

These results apply to the models listed at the start of this notebook. The broader validation comparisons are in 03 and 05; they need unused dates for a fresh independent evaluation.